# Section 03: Designing for Security and Compliance

**Companion notebook for**: `03-security-compliance.html`

## Learning Objectives
- Configure IAM policies and service accounts programmatically
- Manage Cloud KMS keys and encryption
- Set up VPC Service Controls
- Work with Cloud Armor security policies
- Implement compliance controls (audit logging, org policies)

## Prerequisites
- A Google Cloud project with billing enabled
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q google-cloud-iam google-cloud-kms google-cloud-resource-manager google-cloud-dlp

In [ ]:
import os, json
PROJECT_ID = os.environ.get('GCP_PROJECT_ID', 'your-project-id')
REGION = 'us-central1'
print(f'Project: {PROJECT_ID}\nRegion: {REGION}')
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

---
## Section 1: IAM Role Comparison

Understand role types and the principle of least privilege.

In [ ]:
iam_roles = [
    {'type': 'Basic', 'example': 'roles/editor', 'scope': 'All resources', 'production': 'NEVER', 'reason': 'Too broad -- grants write to everything'},
    {'type': 'Predefined', 'example': 'roles/run.developer', 'scope': 'Cloud Run', 'production': 'DEFAULT', 'reason': 'Google-maintained, least-privilege aligned'},
    {'type': 'Custom', 'example': 'projects/x/roles/dataReader', 'scope': 'Custom', 'production': 'SOMETIMES', 'reason': 'When no predefined role matches exactly'},
]

print('IAM Role Types for PCA Exam')
print('=' * 90)
for r in iam_roles:
    print(f"\n  Type: {r['type']}")
    print(f"  Example: {r['example']}")
    print(f"  Production Use: {r['production']}")
    print(f"  Reason: {r['reason']}")

print('\n\nCommon Predefined Roles by Service:')
roles_by_service = {
    'Cloud Run': ['roles/run.developer', 'roles/run.invoker', 'roles/run.admin'],
    'GKE': ['roles/container.developer', 'roles/container.admin', 'roles/container.clusterViewer'],
    'Cloud SQL': ['roles/cloudsql.client', 'roles/cloudsql.editor', 'roles/cloudsql.admin'],
    'BigQuery': ['roles/bigquery.dataViewer', 'roles/bigquery.dataEditor', 'roles/bigquery.jobUser'],
    'Cloud Storage': ['roles/storage.objectViewer', 'roles/storage.objectCreator', 'roles/storage.admin'],
}
for svc, roles in roles_by_service.items():
    print(f'\n  {svc}:')
    for role in roles:
        print(f'    - {role}')

---
## Section 2: Service Account Best Practices

In [ ]:
sa_commands = [
    ('Create dedicated SA', 'gcloud iam service-accounts create cloud-run-sa --display-name="Cloud Run SA"'),
    ('Grant specific role', 'gcloud projects add-iam-policy-binding PROJECT_ID --member="serviceAccount:cloud-run-sa@PROJECT_ID.iam.gserviceaccount.com" --role="roles/cloudsql.client"'),
    ('Deploy with SA', 'gcloud run deploy my-api --service-account=cloud-run-sa@PROJECT_ID.iam.gserviceaccount.com --image=gcr.io/PROJECT_ID/api:v1'),
    ('Workload Identity', 'gcloud iam service-accounts add-iam-policy-binding SA@PROJECT.iam.gserviceaccount.com --role="roles/iam.workloadIdentityUser" --member="serviceAccount:PROJECT.svc.id.goog[NAMESPACE/KSA]"'),
]

print('Service Account Setup Commands')
print('=' * 60)
for desc, cmd in sa_commands:
    print(f'\n## {desc}')
    print(f'   {cmd}')

print('\n\nAnti-Patterns to AVOID:')
anti_patterns = [
    'Using default compute service account in production',
    'Exporting service account keys (use Workload Identity instead)',
    'Granting roles/editor to a service account',
    'Sharing one SA across multiple unrelated services',
    'Not rotating keys if they must exist',
]
for ap in anti_patterns:
    print(f'  X  {ap}')

---
## Section 3: Encryption and Cloud KMS

In [ ]:
encryption_layers = [
    ('Default (Google-managed)', 'All data at rest', 'Google', 'None required'),
    ('CMEK', 'Data at rest + key control', 'Customer (Cloud KMS)', 'Create key, configure service'),
    ('CSEK', 'CE disks only', 'Customer (external)', 'Provide key with API calls'),
    ('Client-side', 'Before upload', 'Customer (app-level)', 'Encrypt in application code'),
]

print(f"{'Layer':<28} {'Protects':<25} {'Key Owner':<25} {'Setup'}")
print('-' * 100)
for layer in encryption_layers:
    print(f'{layer[0]:<28} {layer[1]:<25} {layer[2]:<25} {layer[3]}')

print('\n\nCloud KMS Commands:')
kms_cmds = [
    'gcloud kms keyrings create my-keyring --location=us-central1',
    'gcloud kms keys create my-key --keyring=my-keyring --location=us-central1 --purpose=encryption --rotation-period=90d',
    'gcloud sql instances create my-db --disk-encryption-key=projects/P/locations/us-central1/keyRings/my-keyring/cryptoKeys/my-key',
]
for cmd in kms_cmds:
    print(f'  {cmd}')

---
## Section 4: VPC Service Controls

In [ ]:
print('VPC Service Controls Architecture')
print('=' * 50)
print('''
+--------------------------------------------------+
|              Service Perimeter                     |
|                                                    |
|   +----------+    +-----------+    +----------+   |
|   | Project A |    | Project B |    | Project C|   |
|   | BigQuery  |    | GCS       |    | Vertex AI|   |
|   +----------+    +-----------+    +----------+   |
|                                                    |
|   Restricted: storage.googleapis.com               |
|               bigquery.googleapis.com              |
|               aiplatform.googleapis.com             |
|                                                    |
+--------------------------------------------------+
       |                                    |
  Access Level: Corp VPN          Ingress Rule: Partner ID
  (IP: 203.0.113.0/24)           (specific API methods)
''')

vpcsc_cmds = [
    ('Create access policy', 'gcloud access-context-manager policies create --organization=ORG_ID --title="Prod Security"'),
    ('Create access level', 'gcloud access-context-manager levels create corp-vpn --policy=POLICY_ID --basic-level-spec=level.yaml'),
    ('Create perimeter', 'gcloud access-context-manager perimeters create prod --policy=POLICY_ID --resources="projects/123" --restricted-services="storage.googleapis.com,bigquery.googleapis.com"'),
]
for desc, cmd in vpcsc_cmds:
    print(f'## {desc}')
    print(f'   {cmd}\n')

---
## Section 5: Cloud Armor WAF Policies

In [ ]:
armor_rules = [
    ('Geo-block', '1000', "origin.region_code == 'CN'", 'deny-403'),
    ('SQL injection', '2000', "evaluatePreconfiguredWaf('sqli-v33-stable')", 'deny-403'),
    ('XSS protection', '2100', "evaluatePreconfiguredWaf('xss-v33-stable')", 'deny-403'),
    ('Rate limit', '3000', 'true (all traffic)', 'throttle: 100 req/min'),
    ('Bot detection', '4000', "evaluatePreconfiguredWaf('rce-v33-stable')", 'deny-403'),
]

print('Cloud Armor Security Policy Rules')
print('=' * 80)
print(f"{'Rule':<18} {'Priority':<10} {'Expression':<45} {'Action'}")
print('-' * 80)
for name, pri, expr, action in armor_rules:
    print(f'{name:<18} {pri:<10} {expr:<45} {action}')

print('\nRemember: Cloud Armor ONLY works with External HTTP(S) Load Balancer!')

---
## Section 6: Compliance Checklist

In [ ]:
compliance = {
    'HIPAA': [
        'Sign BAA with Google Cloud',
        'Use only BAA-covered services for PHI',
        'Enable CMEK for PHI at rest',
        'Enable Data Access audit logs',
        'Create VPC-SC perimeter around PHI projects',
        'Restrict IAM to minimum necessary',
    ],
    'SOC 2': [
        'Least-privilege IAM (no basic roles)',
        'Enable audit logging (Admin + Data Access)',
        'Organization policy constraints',
        'SCC Premium for compliance monitoring',
        'Regular access reviews',
    ],
    'GDPR / Data Sovereignty': [
        'Resource location constraint (org policy)',
        'Regional resources (not multi-region)',
        'Assured Workloads for EU sovereignty',
        'VPC-SC to prevent data exfiltration',
        'Right to deletion procedures',
    ],
}

for framework, items in compliance.items():
    print(f'\n{framework} Compliance Checklist')
    print('=' * 45)
    for i, item in enumerate(items, 1):
        print(f'  [ ] {i}. {item}')

---
## Section 7: Security Command Center

In [ ]:
scc_features = {
    'Feature': ['Security Health Analytics', 'Event Threat Detection', 'Container Threat Detection', 'Web Security Scanner', 'Compliance Monitoring', 'Attack Path Simulation'],
    'Standard': ['Basic misconfigs', 'No', 'No', 'Basic', 'No', 'No'],
    'Premium': ['140+ detectors', 'Yes (log-based)', 'Yes (GKE runtime)', 'Managed + auth', 'CIS, PCI, NIST', 'Yes'],
}

print(f"{'Feature':<30} {'Standard (Free)':<20} {'Premium'}")
print('-' * 70)
for i, feat in enumerate(scc_features['Feature']):
    print(f"{feat:<30} {scc_features['Standard'][i]:<20} {scc_features['Premium'][i]}")

print('\nKey takeaway: SCC Premium is required for compliance monitoring and threat detection.')

---
## Summary

This notebook covered PCA Section 03:

1. **IAM Roles** -- Basic vs Predefined vs Custom, common roles by service
2. **Service Accounts** -- Creation, Workload Identity, anti-patterns
3. **Encryption** -- Default, CMEK, CSEK, client-side; Cloud KMS commands
4. **VPC Service Controls** -- Perimeters, access levels, ingress/egress
5. **Cloud Armor** -- WAF rules, geo-blocking, rate limiting
6. **Compliance** -- HIPAA, SOC 2, GDPR checklists
7. **SCC** -- Standard vs Premium tier comparison

**Next**: Section 04 covers analyzing and optimizing processes.